# 02 · ¿Dónde vive "banco"? 
### Embeddings contextuales

Notebook que acompaña el post de LinkedIn. Medimos con BERT en español (BETO) que
la misma palabra en dos contextos produce **dos vectores distintos** -la limitación
estructural de word2vec (un vector por palabra, siempre) que los Transformers resolvieron-.

📄 Referencias:
- Vaswani et al. (2017). *Attention is All You Need*. → [arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)
- Devlin et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. → [arxiv.org/abs/1810.04805](https://arxiv.org/abs/1810.04805)
- Cañete et al. (2020). *BETO: Spanish BERT*. → [github.com/dccuchile/beto](https://github.com/dccuchile/beto)


In [ ]:
%pip install transformers torch scikit-learn -q

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL = "dccuchile/bert-base-spanish-wwm-uncased"   # BETO, ~420MB la primera vez
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL)
model.eval()
print("BETO cargado")

## Extraer el vector de UNA palabra en UNA frase

Con BERT, el embedding de "banco" sale de `last_hidden_state` en la posición del token,
y ese vector ya "miró" al resto de la oración vía atención.

In [3]:
def vector_de(palabra, frase):
    """Devuelve el embedding contextual de `palabra` dentro de `frase`."""
    enc = tok(frase, return_tensors="pt")
    with torch.no_grad():
        out = model(**enc).last_hidden_state[0]          # (n_tokens, 768)
    tokens = tok.convert_ids_to_tokens(enc["input_ids"][0])
    idx = [i for i, t in enumerate(tokens) if palabra in t.lower()]
    assert idx, f"'{palabra}' no aparece tokenizada en: {tokens}"
    return out[idx[0]].numpy(), tokens

frases = {
    "banco_plaza":   "me senté en un banco de la plaza a descansar",
    "banco_credito": "fui al banco a pedir un crédito para la casa",
    "banco_plaza2":  "el banco de madera de la plaza estaba roto",
    "banco_credito2":"el banco me aprobó el préstamo con un interés altísimo",
}

vectores = {k: vector_de("banco", f)[0] for k, f in frases.items()}
print("4 vectores de 'banco' extraídos, uno por contexto")

4 vectores de 'banco' extraídos, uno por contexto


## El momento de la verdad: similitud entre los cuatro "banco"

Si word2vec tuviera razón, las cuatro similitudes serían ~1.0 (mismo punto).
Si el contexto define el vector, los "banco" del mismo sentido deberían parecerse
entre sí **mucho más** que entre sentidos distintos.

In [4]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

nombres = list(vectores)
M = cosine_similarity(np.stack([vectores[n] for n in nombres]))
pd.DataFrame(M, index=nombres, columns=nombres).round(3)

,banco_plaza,banco_credito,banco_plaza2,banco_credito2
banco_plaza,1.000,0.475,0.646,0.412
banco_credito,0.475,1.000,0.496,0.676
banco_plaza2,0.646,0.496,1.000,0.453
banco_credito2,0.412,0.676,0.453,1.000


In [ ]:
intra_plaza   = M[0, 2]   # plaza ↔ plaza2
intra_credito = M[1, 3]   # credito ↔ credito2
inter         = np.mean([M[0, 1], M[0, 3], M[2, 1], M[2, 3]])

print(f"mismo sentido (plaza↔plaza):     {intra_plaza:.3f}")
print(f"mismo sentido (crédito↔crédito): {intra_credito:.3f}")
print(f"sentidos distintos (promedio):   {inter:.3f}")
print()
print("→ La MISMA palabra vive en puntos distintos según su contexto.")
print("  Eso es lo que el mecanismo de atención le agregó al NLP.")

mismo sentido (plaza↔plaza):     0.646
mismo sentido (crédito↔crédito): 0.676
sentidos distintos (promedio):   0.459

→ La MISMA palabra vive en puntos distintos según su contexto.
  Eso es lo que el mecanismo de atención le agregó al NLP.


## Conclusiones

- Esto es la base de por qué los modelos de embeddings modernos (sentence-transformers, SPECTER2) embeben **oraciones y documentos completos**, no palabras sueltas.
- La desambiguación que se ve acá es la que hace que un buscador semántico entienda "abrir una cuenta en el banco" y no devuelva bancos de plazas.
